# Silver Layer: Airline-Route Table Cleaning & Validation

Cleanses `skyledger_airlines.bronze.airline_route` into `skyledger_airlines.silver.airline_route`.
Applies primary key validation (§3.1), numeric range validation (§3.2), and referential integrity checks (§3.4).
Invalid records are quarantined in `silver.airline_route_quarantine` with comma-separated rejection reasons.
Includes Bronze-Silver-Quarantine reconciliation and a data-quality report.

**Relationships:** `airline_route` joins `airline` and `route` (which carriers operate which routes).
FK `ar_airlineid` → `silver.airline.al_airlineid`; FK `ar_routeid` → `silver.route.ro_routeid`.

**Composite PK:** `ar_airlineid` + `ar_routeid` — both must be non-null and the pair must be unique.
**Numeric rule (§3.2):** `ar_operatingcost` must be strictly greater than zero.

In [0]:
%sql
select * from skyledger_airlines.bronze.airline_route

In [0]:
# ── Step 1: Read Bronze, Cast Types, Drop Metadata Columns ──
from pyspark.sql.functions import col, when, lit, count as _count, concat_ws
from pyspark.sql.window import Window

bronze_ar = spark.table("skyledger_airlines.bronze.airline_route")

# Drop metadata columns: _ingestion_timestamp, _source_file, _pipeline_run_id
# Cast composite key and FK columns to INT; operating cost to DOUBLE
ar_typed = bronze_ar.select(
    col("ar_airlineid").cast("int").alias("ar_airlineid"),
    col("ar_routeid").cast("int").alias("ar_routeid"),
    col("ar_available_seats").cast("int").alias("ar_available_seats"),
    col("ar_operatingcost").cast("double").alias("ar_operatingcost"),
    col("ar_comment").cast("string").alias("ar_comment"),
)

bronze_count = ar_typed.count()
print(f"Bronze airline_route rows read: {bronze_count}")
display(ar_typed.limit(10))

In [0]:
# ── Step 2: Primary Key Validation ──
# Rule: Composite key (ar_airlineid + ar_routeid) must be NOT NULL and UNIQUE.
# Failure codes: NULL_PK, DUPLICATE_PK

from pyspark.sql.functions import row_number

ar_flagged = ar_typed.withColumn(
    "is_null_pk",
    when(col("ar_airlineid").isNull() | col("ar_routeid").isNull(), lit(True)).otherwise(lit(False)),
)

# Window for duplicate detection and selection
pk_window = Window.partitionBy("ar_airlineid", "ar_routeid").orderBy(lit(1))

# Assign row_number to keep only one per composite PK
ar_flagged = ar_flagged.withColumn(
    "_pk_count", _count("*").over(Window.partitionBy("ar_airlineid", "ar_routeid"))
).withColumn(
    "is_duplicate_pk",
    when(
        (col("ar_airlineid").isNotNull()) & (col("ar_routeid").isNotNull()) & (col("_pk_count") > 1),
        lit(True),
    ).otherwise(lit(False)),
).withColumn(
    "_pk_row_number", row_number().over(pk_window)
)

# Mark as duplicate except first valid row
ar_flagged = ar_flagged.withColumn(
    "is_duplicate_pk",
    when((col("is_duplicate_pk") == True) & (col("_pk_row_number") > 1), lit(True)).otherwise(col("is_duplicate_pk")),
)

null_pk_count = ar_flagged.filter(col("is_null_pk")).count()
duplicate_pk_count = ar_flagged.filter(col("is_duplicate_pk") == True).count()

print(f"NULL_PK failures:      {null_pk_count}")
print(f"DUPLICATE_PK failures: {duplicate_pk_count}")

In [0]:
# ── Step 3: Numeric Range Validation (§3.2) ──
# Rule: ar_operatingcost must be strictly greater than zero.
# Failure code: INVALID_NUMERIC_OPERATINGCOST

ar_flagged = ar_flagged.withColumn(
    "is_invalid_operatingcost",
    when(col("ar_operatingcost").isNull() | (col("ar_operatingcost") <= 0), lit(True)).otherwise(lit(False)),
)

invalid_operatingcost_count = ar_flagged.filter(col("is_invalid_operatingcost")).count()
print(f"INVALID_NUMERIC_OPERATINGCOST failures: {invalid_operatingcost_count}")

In [0]:
# ── Step 4: Referential Integrity Validation (§3.4) ──
# Rules:
#   ar_airlineid must resolve to a valid al_airlineid in silver.airline → INVALID_FK_AIRLINE
#   ar_routeid must resolve to a valid ro_routeid in silver.route → INVALID_FK_ROUTE
#
# Parent-before-child order: airline and route are parents of airline_route.
# Uses silver.airline and silver.route (already validated) as references.

valid_airline_ids = (
    spark.table("skyledger_airlines.silver.airline")
    .select(col("al_airlineid").cast("int").alias("al_airlineid"))
    .filter(col("al_airlineid").isNotNull())
    .distinct()
)

valid_route_ids = (
    spark.table("skyledger_airlines.silver.route")
    .select(col("ro_routeid").cast("int").alias("ro_routeid"))
    .filter(col("ro_routeid").isNotNull())
    .distinct()
)

# Left join for airline FK
ar_flagged = ar_flagged.join(
    valid_airline_ids,
    ar_flagged["ar_airlineid"] == valid_airline_ids["al_airlineid"],
    "left",
).withColumn(
    "is_invalid_fk_airline",
    when(col("al_airlineid").isNull(), lit(True)).otherwise(lit(False)),
)

# Left join for route FK
ar_flagged = ar_flagged.join(
    valid_route_ids,
    ar_flagged["ar_routeid"] == valid_route_ids["ro_routeid"],
    "left",
).withColumn(
    "is_invalid_fk_route",
    when(col("ro_routeid").isNull(), lit(True)).otherwise(lit(False)),
)

invalid_fk_airline_count = ar_flagged.filter(col("is_invalid_fk_airline")).count()
invalid_fk_route_count = ar_flagged.filter(col("is_invalid_fk_route")).count()
print(f"INVALID_FK_AIRLINE failures: {invalid_fk_airline_count}")
print(f"INVALID_FK_ROUTE failures:   {invalid_fk_route_count}")

In [0]:
# ── Step 5: Build Rejection Reasons, Split Clean vs Quarantine, Write Tables ──

# Build comma-separated rejection_reason from all flag columns
ar_flagged = ar_flagged.withColumn(
    "rejection_reason",
    concat_ws(
        ", ",
        when(col("is_null_pk"), lit("NULL_PK")),
        when(col("is_duplicate_pk"), lit("DUPLICATE_PK")),
        when(col("is_invalid_operatingcost"), lit("INVALID_NUMERIC_OPERATINGCOST")),
        when(col("is_invalid_fk_airline"), lit("INVALID_FK_AIRLINE")),
        when(col("is_invalid_fk_route"), lit("INVALID_FK_ROUTE")),
    ),
)

# Clean records: passed every check (rejection_reason is empty string)
silver_ar = ar_flagged.filter(col("rejection_reason") == "").select(
    "ar_airlineid", "ar_routeid", "ar_available_seats", "ar_operatingcost", "ar_comment"
)

# Quarantined records: failed one or more checks
quarantine_ar = ar_flagged.filter(col("rejection_reason") != "").select(
    "ar_airlineid", "ar_routeid", "ar_available_seats", "ar_operatingcost", "ar_comment",
    "rejection_reason",
)

silver_count = silver_ar.count()
quarantine_count = quarantine_ar.count()

print(f"Silver rows:     {silver_count}")
print(f"Quarantine rows: {quarantine_count}")

# Write Silver table (overwrite for idempotency — same result on every run)
(silver_ar.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.airline_route"))

# Write Quarantine table (overwrite for idempotency)
(quarantine_ar.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.airline_route_quarantine"))

print("✅ Tables written: silver.airline_route, silver.airline_route_quarantine")

In [0]:
# ── Step 6: Bronze-Silver-Quarantine Reconciliation (§3.5) ──
# Bronze row count must equal Silver + Quarantine for the airline_route table.

bronze_total = spark.table("skyledger_airlines.bronze.airline_route").count()
silver_total = spark.table("skyledger_airlines.silver.airline_route").count()
quarantine_total = spark.table("skyledger_airlines.silver.airline_route_quarantine").count()

print(f"Bronze row count:      {bronze_total}")
print(f"Silver row count:      {silver_total}")
print(f"Quarantine row count:  {quarantine_total}")
print(f"Silver + Quarantine:   {silver_total + quarantine_total}")

assert bronze_total == silver_total + quarantine_total, (
    f"RECONCILIATION FAILED: Bronze ({bronze_total}) != "
    f"Silver ({silver_total}) + Quarantine ({quarantine_total})"
)

print("✅ Reconciliation PASSED: Bronze = Silver + Quarantine")

In [0]:
# ── Step 7: Data Quality Report ──
# Documents every check executed, even when zero rows fail.

dq_report = spark.createDataFrame(
    [
        ("airline_route", "NULL_PK", "Composite key (ar_airlineid + ar_routeid) must be fully populated", null_pk_count, "Quarantine"),
        ("airline_route", "DUPLICATE_PK", "Composite key (ar_airlineid + ar_routeid) must be unique", duplicate_pk_count, "Quarantine"),
        ("airline_route", "INVALID_NUMERIC_OPERATINGCOST", "ar_operatingcost must be strictly greater than zero", invalid_operatingcost_count, "Quarantine"),
        ("airline_route", "INVALID_FK_AIRLINE", "ar_airlineid must reference a valid al_airlineid in silver.airline", invalid_fk_airline_count, "Quarantine"),
        ("airline_route", "INVALID_FK_ROUTE", "ar_routeid must reference a valid ro_routeid in silver.route", invalid_fk_route_count, "Quarantine"),
    ],
    ["table", "rule_code", "check_description", "failing_rows", "action_taken"],
)

display(dq_report)

# Show quarantined records if any exist
if quarantine_count > 0:
    print(f"\n── Quarantined Records ({quarantine_count} rows) ──")
    display(spark.table("skyledger_airlines.silver.airline_route_quarantine"))
else:
    print("\nNo quarantined records — all airline_route rows passed every check.")

# Show final Silver table
print(f"\n── Silver Airline-Route Table ({silver_count} rows) ──")
display(spark.table("skyledger_airlines.silver.airline_route"))